In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np

import holidays
import tqdm
import os

import sklearn
sklearn.set_config(transform_output='pandas')

In [ ]:
import helper as hl
import prep_utils

In [7]:
DATASET = 'paloalto'   # dundee | porto | boulder | paloalto

SR_FREQ = '12H'
MIN_POINTS = 100

LOOKAHEAD_STEPS = 1
DATA_PATH = os.path.join('.', 'data')

EPS = 1e-9

# Processing / Feature Engineering

In [8]:
df_evse_meta = pd.read_pickle(f'./data/pkl/{DATASET}_data.metadata.v3.pickle')

# Get Location (lon, lat) of evse locations (just in case)
df_evse_meta.loc[:,'lon'] = df_evse_meta.geometry.x
df_evse_meta.loc[:,'lat'] = df_evse_meta.geometry.y

if (DATASET == 'porto') or (DATASET == 'boulder') or (DATASET == 'paloalto'):
    df_evse_meta.rename(
        {
            'building':'Site',
            'evse_type':'Model'
        },
        axis=1,
        inplace=True
    )

In [9]:
df_evse_demand = pd.read_pickle(
    os.path.join(
        DATA_PATH, 'pkl', 
        f"{DATASET}_data.demand_{SR_FREQ}_{MIN_POINTS}_points.v4.pickle"
    )
).reset_index()

#   # Rename columns to more generic attributes
df_evse_demand = df_evse_demand.rename({
    'evse_id':'oid', 
    'time_axis':'timestamp', 
    f'kW_{SR_FREQ}':'power_curr', 
}, axis=1)

In [10]:
#   # Add ```is_holiday``` flag
if DATASET == 'dundee':
    uk_holidays = holidays.country_holidays('UK', subdiv='SCT')
    df_evse_demand.loc[:, 'is_holiday'] = df_evse_demand['timestamp'].apply(lambda l: l in uk_holidays)    
    
elif DATASET == 'porto':
    pt_holidays = holidays.country_holidays(country='PT', subdiv=13)  # Subdivision: '13' (Porto)
    df_evse_demand.loc[:, 'is_holiday'] = df_evse_demand['timestamp'].apply(lambda l: l in pt_holidays)    

elif DATASET == 'boulder':
    us_holidays = holidays.country_holidays(country='US', subdiv='CO')  # Subdivision: 'CO' (Colorado)
    df_evse_demand.loc[:, 'is_holiday'] = df_evse_demand['timestamp'].apply(lambda l: l in us_holidays)    

elif DATASET == 'paloalto':
    us_holidays = holidays.country_holidays(country='US', subdiv='CA')  # Subdivision: 'CA' (California)
    df_evse_demand.loc[:, 'is_holiday'] = df_evse_demand['timestamp'].apply(lambda l: l in us_holidays)    

In [11]:
#   # Add Temporal-based Information (from timestamp)
df_evse_demand.loc[:, 'hour'] = df_evse_demand['timestamp'].dt.hour
df_evse_demand.loc[:, 'day'] = df_evse_demand['timestamp'].dt.day_of_week
df_evse_demand.loc[:,'week'] = df_evse_demand['timestamp'].dt.isocalendar()['week']
df_evse_demand.loc[:,'month'] = df_evse_demand['timestamp'].dt.month

#   # Cyclical Encoding of Temporal-based Information
df_evse_demand['hour_sin'] = np.sin(2 * np.pi * df_evse_demand['hour'] / 24)
df_evse_demand['hour_cos'] = np.cos(2 * np.pi * df_evse_demand['hour'] / 24)

df_evse_demand['day_sin'] = np.sin(2 * np.pi * df_evse_demand['day'] / 7)
df_evse_demand['day_cos'] = np.cos(2 * np.pi * df_evse_demand['day'] / 7)

df_evse_demand['week_sin'] = np.sin(2 * np.pi * df_evse_demand['week'] / 52)
df_evse_demand['week_cos'] = np.cos(2 * np.pi * df_evse_demand['week'] / 52)

df_evse_demand['month_sin'] = np.sin(2 * np.pi * df_evse_demand['month'] / 12)
df_evse_demand['month_cos'] = np.sin(2 * np.pi * df_evse_demand['month'] / 12)

In [12]:
# Fill NaN values at ```charging_time```
df_evse_demand[['charging_time']] = df_evse_demand[['charging_time']].fillna(value=0).astype(float)

In [13]:
#   # Add Extrapolated Forecast
df_evse_demand.loc[:, 'power_curr_lag1'] = df_evse_demand.groupby('oid')['power_curr'].shift(1).fillna(0) # Energy demand at t-1

tqdm.tqdm.pandas(desc='Calculating extrapolated forecast(s)...')
df_evse_demand = df_evse_demand.join(
    df_evse_demand.dropna().progress_apply(
        lambda l: pd.Series(
            prep_utils.extrapolate_energy_demand(
                [
                    l['power_curr_lag1'], 
                    l['power_curr']
                ], 
                freq=SR_FREQ.lower(),
                steps_ahead=(steps_ahead:=1)
            ),
            index=[f'power_next_step{i+1}_extrap' for i in range(steps_ahead)]
        ), 
        axis=1
    )
)

Calculating extrapolated forecast(s)...: 100%|██████████| 127384/127384 [00:31<00:00, 4034.46it/s]


In [14]:
# Add delta from prev (autoregressive feature)
df_evse_demand.loc[:, 'power_curr_delta'] = df_evse_demand['power_curr'] - df_evse_demand['power_curr_lag1']
# df_evse_demand.loc[:, 'power_curr_pct_delta'] =  ((df_evse_demand['power_curr'] - df_evse_demand['power_curr_lag1']) / df_evse_demand['power_curr_lag1']) * 100
df_evse_demand.loc[:, "power_curr_logdelta"] = np.log(df_evse_demand['power_curr'] + EPS) - np.log(df_evse_demand['power_curr_lag1'] + EPS)

#   # Add evse-related information
#   # ...Add building location to main dataset
location_lookup = df_evse_meta['Site'].to_dict()
df_evse_demand.loc[:, 'building'] = df_evse_demand['oid'].apply(lambda l: location_lookup[l] if l in location_lookup else '<UNK>')

#   # ...Add charger model to main dataset
model_lookup = df_evse_meta['Model'].to_dict()
df_evse_demand.loc[:, 'model'] = df_evse_demand['oid'].apply(lambda l: model_lookup[l] if l in model_lookup else '<UNK>')

#   # ...Add evse power output and number of outlets
df_evse_demand = pd.merge(
    df_evse_demand, 
    df_evse_meta.loc[
        :, 
        df_evse_meta.columns.intersection(
            ['Site', 'power_outlets', 'power_output_kW', 'lon', 'lat']
        )
    ], 
    left_on='oid', 
    right_index=True, 
    how='left'
)

In [15]:
df_evse_demand = df_evse_demand.sort_values(
    'timestamp'
).set_index(
    'oid',
    append=True
)

In [16]:
# Get Percentage of Inactivity (zero consumption)
df_evse_demand[f'is_offline'] = np.isclose(
    df_evse_demand.power_curr, 0, atol=3e-1    # < 2-3 kW over 12H = Standby/background consumption
).astype(int)

In [17]:
# Calculate downtime/uptime between demand
df_evse_demand[f'downtime'] = df_evse_demand.groupby(
    level=1
).apply(
    lambda l: hl.reset_cumsum(l[f'is_offline'], lambda a,b: a+b if b != 0 else 0)
).reset_index(level=0, drop=True).astype(float)

df_evse_demand[f'uptime'] = df_evse_demand.groupby(
    level=1
).apply(
    lambda l: hl.reset_cumsum(1-l[f'is_offline'], lambda a,b: a+b if b != 0 else 0)
).reset_index(level=0, drop=True).astype(float)

# Calculate activity score
df_evse_demand['activity_score'] = np.exp(-df_evse_demand['downtime'])

In [ ]:
# Save for future use
df_evse_demand = df_evse_demand.set_index('timestamp', append=True).sort_index()

df_evse_demand.to_pickle(
    os.path.join(
        DATA_PATH, 'pkl', 
        f"{DATASET}_data.demand_{SR_FREQ}_{MIN_POINTS}_points.enriched.v4.pickle"
    )
)